## Expert Knowledge Worker

### A question answering agent that is an expert knowledge worker
### To be used by employees of Insurellm, an Insurance Tech company
### The agent needs to be accurate and the solution should be low cost.

This project will use RAG (Retrieval Augmented Generation) to ensure our question/answering assistant has high accuracy.

In [2]:
# imports

import os
import glob
from dotenv import load_dotenv
import gradio as gr

In [4]:
# show the interpreter the notebook is actually using
import sys
print(sys.executable)
print(sys.path[:3])

/mnt/d/GitHub/llm-engineering/llm-eng/bin/python
['/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload']


In [ ]:
import langchain, os, pkgutil
print("langchain:", langchain.__file__)
print("version:", getattr(langchain, "__version__", "unknown"))
print("top-level submodules:", [m.name for m in pkgutil.iter_modules([os.path.dirname(langchain.__file__)])])

langchain: /mnt/d/GitHub/llm-engineering/llm-eng/lib/python3.12/site-packages/langchain/__init__.py
version: 1.0.3
top-level submodules: ['agents', 'chat_models', 'embeddings', 'messages', 'rate_limiters', 'tools']


In [ ]:
# imports for langchain (robust fallback)
try:
    # try multiple common import styles
    try:
        from langchain.document_loaders.directory import DirectoryLoader
        from langchain.document_loaders.text import TextLoader
    except Exception:
        from langchain.document_loaders import DirectoryLoader, TextLoader  # older layouts
    from langchain.text_splitter import CharacterTextSplitter
except Exception as e:
    print("langchain loaders not available (using local fallback):", e)
    from typing import List, NamedTuple

    # Minimal Document replacement (self-contained)
    class Document(NamedTuple):
        page_content: str
        metadata: dict

    class TextLoader:
        def __init__(self, path, encoding='utf-8'):
            self.path = path
            self.encoding = encoding

        def load(self) -> List[Document]:
            with open(self.path, 'r', encoding=self.encoding) as f:
                return [Document(page_content=f.read(), metadata={"source": self.path})]

    class DirectoryLoader:
        def __init__(self, folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs=None):
            self.folder = folder
            self.glob = glob
            self.loader_cls = loader_cls
            self.loader_kwargs = loader_kwargs or {}

        def load(self) -> List[Document]:
            import glob as _glob, os as _os
            pattern = _os.path.join(self.folder, self.glob)
            files = _glob.glob(pattern, recursive=True)
            docs: List[Document] = []
            for p in files:
                loader = self.loader_cls(p, **self.loader_kwargs)
                docs.extend(loader.load())
            return docs

    class CharacterTextSplitter:
        def __init__(self, chunk_size=1000, chunk_overlap=200):
            self.chunk_size = chunk_size
            self.chunk_overlap = chunk_overlap

        def split_documents(self, documents: List[Document]) -> List[Document]:
            out: List[Document] = []
            for d in documents:
                text = d.page_content or ""
                i = 0
                step = max(1, self.chunk_size - self.chunk_overlap)
                while i < len(text):
                    chunk = text[i:i + self.chunk_size]
                    meta = dict(getattr(d, "metadata", {}))
                    out.append(Document(page_content=chunk, metadata=meta))
                    i += step
            return out

langchain loaders not available (using local fallback): No module named 'langchain.document_loaders'


In [14]:
# price is a factor for our company, so we're going to use a low cost model

MODEL = "gpt-4o-mini"
db_name = "vector_db"

In [15]:
# Load environment variables in a file called .env

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')

In [16]:
# Read in documents using LangChain's loaders
# Take everything in all the sub-folders of our knowledgebase
# Thank you Mark D. and Zoya H. for fixing a bug here..

folders = glob.glob("knowledge-base/*")

# With thanks to CG and Jon R, students on the course, for this fix needed for some users 
text_loader_kwargs = {'encoding': 'utf-8'}
# If that doesn't work, some Windows users might need to uncomment the next line instead
# text_loader_kwargs={'autodetect_encoding': True}

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs=text_loader_kwargs)
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

In [17]:
len(documents)

31

In [18]:
documents[24]

Document(page_content='# HR Record\n\n# Oliver Spencer\n\n## Summary\n- **Date of Birth**: May 14, 1990  \n- **Job Title**: Backend Software Engineer  \n- **Location**: Austin, Texas  \n\n## Insurellm Career Progression\n- **March 2018**: Joined Insurellm as a Backend Developer I, focusing on API development for customer management systems.\n- **July 2019**: Promoted to Backend Developer II after successfully leading a team project to revamp the claims processing system, reducing response time by 30%.\n- **June 2021**: Transitioned to Backend Software Engineer with a broader role in architecture and system design, collaborating closely with the DevOps team.\n- **September 2022**: Assigned as the lead engineer for the new "Innovate" initiative, aimed at integrating AI-driven solutions into existing products.\n- **January 2023**: Awarded a mentorship role to guide new hires in backend technology and best practices within Insurellm.\n\n## Annual Performance History\n- **2018**: **3/5** - 

In [19]:
text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

In [20]:
len(chunks)

127

In [21]:
chunks[6]

Document(page_content="ort to the Client via phone, email, and a ticketing system during business hours (Monday to Friday, 9 AM to 5 PM EST).\n\n2. **Training and Onboarding**: Provider will deliver comprehensive onboarding training for up to ten (10) members of the Client's staff to ensure effective use of the Rellm solution.\n\n3. **Updates and Maintenance**: Provider is responsible for providing updates to the Rellm platform to improve functionality and security, at no additional cost to the Client.\n\n4. **Escalation Protocol**: Issues that cannot be resolved at the first level of support will be escalated to the senior support team, ensuring that critical concerns are addressed promptly.\n\n---\n\n**Acceptance of Terms**: By signing below, both parties agree to the Terms, Renewal, Features, and Support outlined in this Agreement.\n\n**Insurellm, Inc.**  \n_____________________________  \nAuthorized Signature   \nDate: ___________________  \n\n**Apex Reinsurance**  \n______________

In [22]:
doc_types = set(chunk.metadata['doc_type'] for chunk in chunks)
print(f"Document types found: {', '.join(doc_types)}")

Document types found: employees, contracts, company, products


In [23]:
for chunk in chunks:
    if 'CEO' in chunk.page_content:
        print(chunk)
        print("_________")

Document(page_content='iries.\n\n3. **Regular Updates:** Insurellm will offer ongoing updates and enhancements to the Homellm platform, including new features and security improvements.\n\n4. **Feedback Implementation:** Insurellm will actively solicit feedback from GreenValley Insurance to ensure Homellm continues to meet their evolving needs.\n\n---\n\n**Signatures:**\n\n_________________________________  \n**[Name]**  \n**Title**: CEO  \n**Insurellm, Inc.**\n\n_________________________________  \n**[Name]**  \n**Title**: COO  \n**GreenValley Insurance, LLC**  \n\n---\n\nThis agreement represents the complete understanding of both parties regarding the use of the Homellm product and supersedes any prior agreements or communications.', metadata={'source': 'knowledge-base/contracts/Contract with GreenValley Insurance for Homellm.md', 'doc_type': 'contracts'})
_________
Document(page_content='urellm 2025-2026 Roadmap, including mobile integration and telematics-based pricing enhancement